In [27]:
import pandas as pd
import numpy as np
import scipy.stats as stats

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve
from sklearn.calibration import calibration_curve


## test pkls are the updated pkls with the new definition of aki (no + stage 1 vs stage 2 & stage 3)
# DATA_SOURCE = "Preop + Intraop"
# output_pkl = '/home/server/Projects/data/AKI/results/tabular_combined_test.pkl'

DATA_SOURCE = "Preop"
output_pkl = '/home/server/Projects/data/AKI/results/tabular_preop_test2.pkl'

# DATA_SOURCE = "Intraop"
# output_pkl = '/home/server/Projects/data/AKI/results/tabular_intraop_test.pkl'

df = pd.read_pickle(output_pkl)

# make names more readable
model_translate = {
    'base': 'base',
    'log_reg': 'Logistic Regression',
    'lin_reg': 'Linear Regression',
    'autogluon': 'AutoGluon',
    'xgb': 'XGBoost',
    'svm': 'SVM',
    'mlp': 'MLP',
    'rf': 'Random Forest',
    'knn': 'KNN',
    'asa': 'ASA',
}
df['model_name'] = df['model_name'].map(model_translate)

In [28]:
df[df['model_name'] == 'base']['y_pred_binary'].values[0][0].shape

IndexError: index 0 is out of bounds for axis 0 with size 0

In [29]:
df[df['model_name'] == 'ASA']['y_prob'].values[0][0].shape

IndexError: index 0 is out of bounds for axis 0 with size 0

In [30]:
df

,balanced_accuracy,roc_auc,precision,recall,y_true,y_pred_binary,y_prob,model_name
0,"[0.8502879444145645, 0.8287356220552928, 0.839...","[0.9244724585785846, 0.9051622915081967, 0.910...","[0.24561403508771928, 0.2276595744680851, 0.22...","[0.8190127970749543, 0.7824497257769653, 0.808...","[[False, False, False, False, False, False, Fa...","[[False, False, False, False, False, True, Fal...","[[0.31805560032671515, 0.046308432866408664, 0...",Logistic Regression


In [31]:
# plot avged curves
def plot_calibration_curves(df):
    plt.figure()

    model_names = df['model_name'].values.tolist()
    model_names.remove('base')
    for model_name in model_names:

        y_true = np.concatenate(df[df['model_name'] == 'base']['y_pred_binary'].values[0])
        y_prob = np.concatenate(df[df['model_name'] == model_name]['y_prob'].values[0])
        
        prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='uniform')


        plt.plot(prob_pred, prob_true, label=f'{model_name}', lw=2)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Ideal Calibration')
    plt.xlabel('Mean predicted probability')
    plt.ylabel('Fraction of positives')
    plt.title(f'Calibration Curve on {DATA_SOURCE} data')
    plt.legend(loc='upper left')
    plt.grid(True)

def plot_roc_curve(df):
    plt.figure()

    model_names = df['model_name'].values.tolist()
    model_names.remove('base')
    for model_name in model_names:
        df_base = df[df['model_name'] == 'base'].iloc[0]
        df_row = df[df['model_name'] == model_name].iloc[0]

        # Set up common FPR points to interpolate all curves onto
        mean_fpr = np.linspace(0, 1, 100)
        tprs = []
        aucs = []

        # Collect TPRs and AUCs from all folds/splits
        for i in range(df_row['y_prob'].shape[0]):
            y_prob = df_row['y_prob'][i]
            y_binary_test = df_base['y_pred_binary'][i]

            fpr, tpr, _ = roc_curve(y_binary_test, y_prob)
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0  # Ensure it starts at 0
            tprs.append(interp_tpr)
            aucs.append(auc(fpr, tpr))

        # Compute the mean and std across interpolated TPRs
        mean_tpr = np.mean(tprs, axis=0)
        std_tpr = np.std(tprs, axis=0)
        mean_auc = auc(mean_fpr, mean_tpr)

        # Plot the averaged ROC curve
        plt.plot(mean_fpr, mean_tpr, label=f'{model_name} (AUC = {mean_auc:.2f})', lw=2)
        # plt.fill_between(mean_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr, color='lightblue', alpha=0.4)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve on {DATA_SOURCE} data')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()

def plot_pr_curve(df):
    plt.figure()

    model_names = df['model_name'].values.tolist()
    model_names.remove('base')
    for model_name in model_names:
        df_base = df[df['model_name'] == 'base'].iloc[0]
        df_row = df[df['model_name'] == model_name].iloc[0]

        mean_recall = np.linspace(0, 1, 100)
        precisions = []
        pr_aucs = []

        for i in range(df_row['y_prob'].shape[0]):
            y_prob = df_row['y_prob'][i]
            y_binary_test = df_base['y_pred_binary'][i]

            precision, recall, _ = precision_recall_curve(y_binary_test, y_prob)
            interp_prec = np.interp(mean_recall, recall[::-1], precision[::-1])  # Make sure recall is increasing
            precisions.append(interp_prec)
            pr_aucs.append(auc(recall, precision))

        mean_prec = np.mean(precisions, axis=0)
        std_prec = np.std(precisions, axis=0)
        mean_pr_auc = auc(mean_recall, mean_prec)

        plt.plot(mean_recall, mean_prec, label=f'{model_name} (AUC = {mean_pr_auc:.2f})', lw=2)
        # plt.fill_between(mean_recall, mean_prec - std_prec, mean_prec + std_prec, alpha=0.4, color='navajowhite')
    plt.plot([0, 1], [1, 0], linestyle='--', color='gray', label='Chance')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve on {DATA_SOURCE} data')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()







plot_calibration_curves(df)
plot_roc_curve(df)
plot_pr_curve(df)

ValueError: list.remove(x): x not in list

<Figure size 640x480 with 0 Axes>

In [23]:
# for model_name in model_names:
#     plot_avg_calibration_curve(df, model_name)
#     plot_avg_roc_curve(df, model_name)
#     plot_avg_pr_curve(df, model_name)

In [24]:
# print out performance metrics

if 'y_pred_binary' in df.columns:
    df = df.drop(columns=['y_pred_binary'])
if 'y_prob' in df.columns:
    df = df.drop(columns=['y_prob'])
model_names = df['model_name'].values.tolist()
model_names.remove('base')
performance_metrics = df.columns.values.tolist()
performance_metrics.remove('model_name')

for model_name in model_names:
    df_row = df[df['model_name'] == model_name].iloc[0]
    print(model_name)
    for performance_metric in performance_metrics:
        arr = df_row[performance_metric]

        mean = np.mean(arr)
        sem = stats.sem(arr)
        confidence = 0.95
        n = len(arr)
        dof = n - 1
        ci = stats.t.interval(confidence, dof, loc=mean, scale=sem)
        print(f'{ci[0]:0.06f}\t{mean:0.06f}\t{ci[1]:0.06f}')
    print()


AutoGluon
0.714269	0.735287	0.756305
0.323854	0.333430	0.343006
0.964013	0.964507	0.965002
0.810163	0.845422	0.880681
0.506645	0.523800	0.540954
0.993535	0.994209	0.994883
0.968999	0.969412	0.969826
0.449605	0.457482	0.465360


TypeError: object of type 'float' has no len()